# Train Factoid SFT: Full Resources, One Gold Alias Per Row

This variant retains all factoid questions and full resources, but expands each training question into one row per official gold alias. Each target contains exactly one `[BE]... [EE]` answer span. The held-out split remains question-level and uses the full gold alias set for generated-dev scoring. LoRA dropout is `0.05`; outputs are isolated from the original full-resource SFT model.

In [ ]:
import json
import sys
from argparse import Namespace
from pathlib import Path

from sklearn.model_selection import train_test_split

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / 'Task-Structured Counterfactual Preference Mining']
project_root = next((path for path in candidates if (path / 'src').is_dir()), None)
assert project_root is not None, 'Run this notebook from the project or workspace directory.'

model_dir = project_root / 'models/Qwen2.5-0.5B-Instruct'
train_file = project_root / 'data/training13b.json'
prepared_dir = project_root / 'data/BioASQ_factoid_sft_prepared/full_resources_per_gold_alias_qwen25_05b'
train_prepared = prepared_dir / 'train_prepared.json'
eval_prepared = prepared_dir / 'eval_prepared.json'
model_output_dir = project_root / 'Artifacts/Factoid_SFT/models/full_resources_per_gold_alias_qwen25_05b_lora_dropout_005'

LORA_DROPOUT = 0.05
SEED = 3407
for path in (model_dir / 'model.safetensors', train_file):
    assert path.exists(), f'Missing required input: {path}'

print(f'Project root: {project_root}')
print(f'Prepared data: {prepared_dir}')
print(f'Model output: {model_output_dir}')

## Inspect the resolved commands

Run this first. It performs no training and shows the exact commands, paths, and parameters.

In [ ]:
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utility.data import load_bioasq_records, save_prepared_records

# Build full-resource prompts once, then split by question before expanding aliases.
prep_args = Namespace(
    question_types=['factoid'], max_resources=0, max_resource_chars=0,
    max_factoid_answers=999, max_summary_answers=1, max_list_items=100,
    resource_granularity='document', resource_selection='first',
    prompt_file='prompts/factoid_single_answer_aligned.json',
    prompt_registry_path=None, prompt='factoid-single-answer-aligned-v1',
)
base_rows = load_bioasq_records(train_file, prep_args)
assert len(base_rows) == 1600, f'Expected 1,600 factoid questions, found {len(base_rows)}'
train_base, eval_base = train_test_split(
    base_rows, test_size=0.1, random_state=SEED, shuffle=True
)
assert len(train_base) == 1440 and len(eval_base) == 160

def gold_aliases(row):
    import re
    aliases = re.findall(r'\[BE\](.*?)\[EE\]', row['output'], flags=re.DOTALL)
    seen = set()
    result = []
    for alias in aliases:
        alias = alias.strip()
        if alias and alias.casefold() not in seen:
            seen.add(alias.casefold())
            result.append(alias)
    return result

def expand_per_alias(rows):
    expanded = []
    for row in rows:
        for alias_index, alias in enumerate(gold_aliases(row)):
            alias_row = dict(row)
            alias_row['source_question_id'] = row['id']
            alias_row['id'] = f"{row['id']}__gold_alias_{alias_index}"
            alias_row['output'] = f'[BE]{alias}[EE]'
            expanded.append(alias_row)
    return expanded

train_rows = expand_per_alias(train_base)
assert train_rows, 'No per-alias training rows were produced.'
assert all(row['output'].count('[BE]') == 1 and row['output'].count('[EE]') == 1 for row in train_rows)

prepared_dir.mkdir(parents=True, exist_ok=True)
save_prepared_records(train_rows, train_prepared)
save_prepared_records(eval_base, eval_prepared)
summary = {
    'source_questions': len(base_rows),
    'train_questions': len(train_base),
    'validation_questions': len(eval_base),
    'per_alias_training_rows': len(train_rows),
    'lora_dropout': LORA_DROPOUT,
}
(prepared_dir / 'preparation_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))

## Train

This resumes automatically if interrupted. It writes only to the new per-alias dropout directory, so the original full-resource adapter is never overwritten.

In [ ]:
import subprocess

command = [
    'python', 'src/utility/answer_gen_ft.py',
    '--train-input', str(train_prepared), '--eval-input', str(eval_prepared),
    '--question-types', 'factoid', '--validation-ratio', '0.0',
    '--prepared-output-dir', str(prepared_dir / 'pipeline_prepared'),
    '--model-name', 'models/Qwen2.5-0.5B-Instruct',
    '--chat-template', 'qwen-2.5', '--prompt-format', 'chat',
    '--prompt-file', 'prompts/factoid_single_answer_aligned.json',
    '--prompt', 'factoid-single-answer-aligned-v1',
    '--max-resources', '0', '--max-resource-chars', '0',
    '--max-factoid-answers', '999', '--max-seq-length', '4096',
    '--per-device-train-batch-size', '1', '--per-device-eval-batch-size', '1',
    '--gradient-accumulation-steps', '32', '--warmup-steps', '5',
    '--num-train-epochs', '6.0', '--learning-rate', '6e-4', '--weight-decay', '0.01',
    '--logging-steps', '1', '--save-strategy', 'steps', '--save-steps', '25',
    '--lora-r', '32', '--lora-alpha', '32', '--lora-dropout', str(LORA_DROPOUT),
    '--dataset-num-proc', '1', '--selection-metric', 'generated_primary_score',
    '--selection-max-seq-length', '3584', '--selection-max-new-tokens', '64',
    '--selection-num-generations', '1', '--selection-aggregation-strategy', 'union',
    '--early-stopping-patience', '3', '--early-stopping-threshold', '0.001',
    '--seed', str(SEED),
    '--run-name', 'bioasq-qwen25-05b-factoid-full-resources-per-gold-alias-dropout-005-sft',
    '--output-dir', str(model_output_dir / 'trainer_output'),
    '--save-model-dir', str(model_output_dir / 'adapter'),
    '--register-alias', 'factoid-sft-full-resources-per-gold-alias-qwen25-05b-dropout-005',
    '--resume-from-checkpoint', 'auto', '--local-files-only',
]
print(' '.join(command))
subprocess.run(command, cwd=project_root, check=True)

## Read the result

After completion, the adapter is in `Artifacts/Factoid_SFT/models/full_resources_per_gold_alias_qwen25_05b_lora_dropout_005/adapter`. Use the test evaluation notebook to compare it with the existing models.